# Computational Theory Assessment

This notebook contains my solutions to the problems in the Computational Theory module.
The problems are based on the SHA-256 hash algorithm, as specified in [FIPS 180-4: Secure Hash Standard (NIST)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

Throughout the notebook references such as *Section 3.2* are referencing the sections in that document.

Each problem has its own section. The section will state the problem, explains the approach taken and
then works through the solution in small code cells.

## Problem 1: Representing SHA-256 Data

### The Problem


> Investigate how the input, output and intermediate data can be represented in Python.
> Explain how you would represent:

> - 32-bit words
> - sequences of 32 bit words
> - input messages
> - 512 bit message blocks
> - the final 256 bit hash value
> - big endian integers

### 1. 32-bit Words

#### What the standard says

[Secure Hash Standard - FIPS PUB 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), *Section 2.1*, defines a words as:

> "A group of either 32 bits (4 bytes) or 64 bits (8 bytes), depending on the secure hash algorithm"

SHA-1, SHA-224 and SHA-256 use 32 bit ( 4 byte) words. SHA-384 and SHA-512 use 64 bit (8byte) words.

Section3.2 defines how words are added. Two words are treated as integers $X$ and $Y$ and the result is
$Z - (X + Y) \bmod 2^{32}$. Any carry out of the top bit is thrown away, so the answer always fits back into 32 bits

A representation of a 32 bit word therefore needs to:

> - hold any value from $0$ to $2^{32} - 1$
> - wrap around modulo $2^{32}$ on addition
> - keep bitwise operations within 32 bits.

#### Python's built in `int`

[The Python docs on Numeric Types]
    (https://docs.python.org/3/builtins/stdtypes.html#numeric-types-int-float-complex)
    state that "integers have unlimited precision". An `int` can store any 32 bit value, but it has no fixed width. It never wraps. it just keeps growing.

The cell below shows this with the largest 32 bit value.

In [1]:
# The largest value that fits in 32 bits: 2**32 - 1, which is 32 one bits.
MAX_WORD = 0xFFFFFFFF

# show the value and how many bits it needs
print(f"{MAX_WORD=} bits needed: {MAX_WORD.bit_length()}")

# Add 1. A 32 bit word should wrap to 0, but a python int does not
x = MAX_WORD + 1

# The result needs 33 bits, so it is no longer a valid word.
print(f"{x=} bits needed: {x.bit_length()}")

MAX_WORD=4294967295 bits needed: 32
x=4294967296 bits needed: 33


#### Option A: `int` with a mask

One option is to keep using `int` and cut the result back to 32 bits after every
operation with a [bitwise AND](https://docs.python.org/3/builtins/stdtypes.html#bitwise-operations-on-integer-types)

`x & 0xFFFFFFFF` keeps only the lowest 32 bits, which is the same as $x \bmod 2^{32}$

In [2]:
# Addition, then mask back to 32 bits. This matches Section 3.2.
print(f"{(MAX_WORD + 1) & MAX_WORD = }")

# Left shifts also grow an int beyond 32 bits.
print(f"{hex(MAX_WORD << 4) = }")

# Masking the shifted value brings it back to   32 bits
print(f"{hex((MAX_WORD << 4) & MAX_WORD) }")

(MAX_WORD + 1) & MAX_WORD = 0
hex(MAX_WORD << 4) = '0xffffffff0'
0xfffffff0


This works. The bad part is that the width is not part of the type. If you forget to mask it, the code still runs but will produce the incorrect hash value. with no error

#### Option B: `numpy.uint32`

NumPy provides [fixed-width integer types](https://numpy.org/doc/stable/iserbasics.types.html)
`numpy.uint32` is a unsigned integer that is always exactly 32 bits wide



In [5]:
# NumPy provides fixed-width integer types such as uint32.
import numpy as np

In [6]:
# Create 32 bit unsigned word holding the largest value
w = np.uint32(MAX_WORD)

# show its type and its size in bytes
print(f"{type(w)=}, {w.itemsize=} bytes")

type(w)=<class 'numpy.uint32'>, w.itemsize=4 bytes
